In [ ]:
import torch
from torch_geometric.data import HeteroData
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv, to_hetero
from torch_geometric.utils import negative_sampling

data = HeteroData()
data['rna'].x = torch.rand(100, 128)
data['ctcf'].x = torch.randn(50, 128)
edge_index = torch.tensor([[0, 1, 1, 2],
                            [0, 5, 2, 3]], dtype=torch.long)
data['rna', 'interacts', 'ctcf'].edge_index = edge_index


data['rna', 'interacts', 'ctcf'].edge_label = torch.ones(edge_index.size(1))

print(data)

class GNNModel(torch.nn.Module):
    def __init__(self, hidden_channels):
        super().__init__()
        self.conv1 = SAGEConv((-1, -1), hidden_channels)
        self.conv2 = SAGEConv((-1, -1), hidden_channels)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        x = self.conv2(x, edge_index)
        return x

model = GNNModel(hidden_channels=64)
model = to_hetero(model, data.metadata(), aggr='sum')

optimizer = torch.optim.Adam(model.parameters(), lr=0.01)


def train():
    model.train()
    optimizer.zero_grad()

    out = model(data.x_dict, data.edge_index_dict)

    edge_idx = data['rna', 'interacts', 'ctcf'].edge_index
    neg_edge_index = negative_sampling(
        edge_index=edge_idx,
        num_nodes=(100, 50)
    )
    all_edges = torch.cat([edge_idx, neg_edge_index], dim=1)

    labels = torch.cat([
    torch.ones(edge_idx.size(1)),
    torch.zeros(neg_edge_index.size(1))
])

    rna_emb  = out['rna'][all_edges[0]]
    ctcf_emb = out['ctcf'][all_edges[1]]
    scores = (rna_emb * ctcf_emb).sum(dim=-1)
    

    loss = F.binary_cross_entropy_with_logits(scores, labels)
    loss.backward()
    optimizer.step()

    return loss.item()


for epoch in range(1, 101):
    loss = train()
    if epoch % 10 == 0:
        print(f'Epoch {epoch:03d}, Loss: {loss:.4f}')

SyntaxError: invalid syntax. Perhaps you forgot a comma? (2502106195.py, line 44)